# PharmaRec - Weight Generator

This notebook generates pre-computed weights for the PharmaRec drug recommendation system.
Upload the resulting artifacts to Hugging Face Hub for fast local deployment.

**Artifacts produced:**
- `cleaned_dataset.csv` — Preprocessed dataset (10K rows)
- `embeddings_cache.pkl` — Sentence-transformer embeddings (10K × 384)
- `collab_stats.pkl` — Collaborative filtering pre-computed stats

## 1. Install Dependencies

In [ ]:
!pip install -q sentence-transformers scikit-surprise pandas nltk kagglehub huggingface_hub numpy scikit-learn

## 2. Configuration

Set your Hugging Face repo name below. The notebook will create it if it doesn't exist.

In [ ]:
# ====== CONFIGURE THIS ======
HF_REPO = "melkamzer/pharmarec-weights"  # Change to your username/repo
TARGET_ROWS = 10000
# =============================

import os
os.makedirs("artifacts", exist_ok=True)
print(f"Artifacts will be saved to ./artifacts/")
print(f"Will upload to HF Hub: {HF_REPO}")

## 3. Download Dataset from Kaggle

If you don't have Kaggle credentials, you can upload the CSV manually in the next cell.

In [ ]:
import pandas as pd
from pathlib import Path

try:
    import kagglehub
    path = Path(kagglehub.dataset_download("mohamedabdelwahabali/drugreview"))
    print(f"Downloaded to: {path}")

    # Combine train/test/validation splits
    parts = []
    for name in ["drug_review_train.csv", "drug_review_test.csv", "drug_review_validation.csv"]:
        f = list(path.glob(name))
        if f:
            df_part = pd.read_csv(f[0])
            parts.append(df_part)
            print(f"  Loaded {name}: {len(df_part):,} rows")

    raw_df = pd.concat(parts, ignore_index=True)
    if len(raw_df) > TARGET_ROWS:
        raw_df = raw_df.head(TARGET_ROWS)
    print(f"\nCombined dataset: {len(raw_df):,} rows")

except Exception as e:
    print(f"Kaggle download failed: {e}")
    print("\n--- ALTERNATIVE: Upload your CSV manually ---")
    print("Upload drug_review.csv or the combined Kaggle CSV to /content/")
    # Uncomment below if uploading manually:
    # from google.colab import files
    # uploaded = files.upload()
    # raw_df = pd.read_csv(list(uploaded.keys())[0])
    # if len(raw_df) > TARGET_ROWS:
    #     raw_df = raw_df.head(TARGET_ROWS)
    # print(f"Loaded {len(raw_df):,} rows from uploaded file")

## 4. Preprocessing Pipeline

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download NLTK data
for res in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(res, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()
PUNCTUATION = string.punctuation

def clean_review(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", PUNCTUATION))
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return " ".join(tokens)

print("Preprocessing pipeline defined.")

In [ ]:
# Rename patient_id to userId if present
if "patient_id" in raw_df.columns:
    raw_df = raw_df.rename(columns={"patient_id": "userId"})

# Drop unnecessary columns
drop_cols = [c for c in ["Unnamed: 0", "review_length"] if c in raw_df.columns]
if drop_cols:
    raw_df = raw_df.drop(columns=drop_cols)

# Handle missing values
before = len(raw_df)
raw_df = raw_df.dropna(subset=["drugName", "review"])
raw_df["condition"] = raw_df["condition"].fillna("Unknown")
raw_df["rating"] = raw_df["rating"].fillna(raw_df["rating"].median())
raw_df["usefulCount"] = raw_df["usefulCount"].fillna(0).astype(int)
print(f"Dropped {before - len(raw_df)} rows with missing critical fields")
print(f"Remaining: {len(raw_df):,} rows")

# Clean review text
print("Cleaning review text (tokenize, remove stops, lemmatize)...")
raw_df["review"] = raw_df["review"].apply(clean_review)

# Create combined_text
raw_df["combined_text"] = raw_df["condition"].astype(str) + " " + raw_df["review"].astype(str)

# Save cleaned dataset
cleaned_path = "artifacts/cleaned_dataset.csv"
raw_df.to_csv(cleaned_path, index=False)
print(f"\nSaved cleaned dataset: {cleaned_path} ({len(raw_df):,} rows, {len(raw_df.columns)} cols)")
raw_df.head()

## 5. Compute Sentence-Transformer Embeddings

In [ ]:
import hashlib
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

# Build feature text: condition doubled + review (matches content.py logic)
cond = raw_df["condition"].fillna("").astype(str)
text = raw_df["combined_text"].fillna("").astype(str)
feature_texts = (cond + ". " + cond + ". " + text).tolist()

print(f"Encoding {len(feature_texts)} texts with {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    feature_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Embeddings shape: {embeddings.shape}")

# Save in the same format as content.py expects
identity = f"rows:{len(feature_texts)}"
key = hashlib.md5(identity.encode("utf-8")).hexdigest()

cache_path = "artifacts/embeddings_cache.pkl"
with open(cache_path, "wb") as f:
    pickle.dump({"key": key, "embeddings": embeddings}, f)

size_mb = os.path.getsize(cache_path) / (1024 * 1024)
print(f"Saved embeddings cache: {cache_path} ({size_mb:.1f} MB)")

## 6. Compute Collaborative Filtering Stats

In [ ]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

df = raw_df.copy()
global_mean = float(df["rating"].mean())

# Per-drug aggregate statistics
stats = (
    df.groupby("drugName")
    .agg(
        count=("rating", "size"),
        mean_rating=("rating", "mean"),
        useful=("usefulCount", "sum"),
        condition=("condition", lambda s: s.mode().iat[0] if not s.mode().empty else "Unknown"),
    )
    .reset_index()
)

# Bayesian-weighted rating
prior_strength = 10.0
m = prior_strength
c = global_mean
stats["weighted_score"] = (
    (stats["count"] / (stats["count"] + m)) * stats["mean_rating"]
    + (m / (stats["count"] + m)) * c
)

drug_condition = dict(zip(stats["drugName"], stats["condition"]))
drug_mean_ratings = dict(zip(stats["drugName"], stats["mean_rating"]))
drug_weighted_scores = dict(zip(stats["drugName"], stats["weighted_score"]))
drug_map = {drug: idx for idx, drug in enumerate(stats["drugName"])}
drug_lookup = {drug.lower(): drug for drug in stats["drugName"]}

# condition -> drugs sorted by weighted score
condition_drugs = {}
for cond_name, grp in stats.sort_values("weighted_score", ascending=False).groupby("condition"):
    condition_drugs[cond_name] = grp["drugName"].tolist()

print(f"Built stats for {len(stats)} drugs across {stats['condition'].nunique()} conditions")

# SVD for RMSE/MAE
print("Fitting SVD for accuracy metrics...")
if "userId" not in df.columns:
    df["userId"] = df.groupby(["drugName", "condition", "review"]).ngroup().astype(str)

ratings_df = pd.DataFrame({
    "user_id": df["userId"].astype(str),
    "item_id": df["drugName"].astype(str),
    "rating": df["rating"].values,
})

lo = min(1.0, float(df["rating"].min()))
hi = max(10.0, float(df["rating"].max()))
reader = Reader(rating_scale=(lo, hi))
data = Dataset.load_from_df(ratings_df, reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
svd_model = SVD()
svd_model.fit(trainset)
preds = svd_model.test(testset)
rmse = float(accuracy.rmse(preds, verbose=False))
mae = float(accuracy.mae(preds, verbose=False))
print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}")

# Save collab stats
collab_stats = {
    "drug_condition": drug_condition,
    "drug_mean_ratings": drug_mean_ratings,
    "drug_weighted_scores": drug_weighted_scores,
    "drug_map": drug_map,
    "drug_lookup": drug_lookup,
    "condition_drugs": condition_drugs,
    "global_mean": global_mean,
    "prior_strength": prior_strength,
    "rmse": rmse,
    "mae": mae,
    "n_drugs": len(stats),
    "n_conditions": int(stats["condition"].nunique()),
    "drug_stats_df": stats.to_dict(orient="records"),
}

collab_path = "artifacts/collab_stats.pkl"
with open(collab_path, "wb") as f:
    pickle.dump(collab_stats, f)

size_mb = os.path.getsize(collab_path) / (1024 * 1024)
print(f"Saved collab stats: {collab_path} ({size_mb:.1f} MB)")

## 7. Upload to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, login

# Login (will prompt for token if not already logged in)
# You can also run: huggingface-cli login in terminal first
login()

api = HfApi()

# Create repo if it doesn't exist
try:
    api.create_repo(HF_REPO, exist_ok=True, repo_type="model")
    print(f"Repo ready: https://huggingface.co/{HF_REPO}")
except Exception as e:
    print(f"Repo setup: {e}")

# Upload files
files_to_upload = [
    ("artifacts/cleaned_dataset.csv", "cleaned_dataset.csv"),
    ("artifacts/embeddings_cache.pkl", "embeddings_cache.pkl"),
    ("artifacts/collab_stats.pkl", "collab_stats.pkl"),
]

for local_path, remote_name in files_to_upload:
    if os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"Uploading {remote_name} ({size_mb:.1f} MB)...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=remote_name,
            repo_id=HF_REPO,
            repo_type="model",
        )
        print(f"  Uploaded {remote_name}")
    else:
        print(f"  Skipping {remote_name} (not found)")

print(f"\nDone! All artifacts uploaded to https://huggingface.co/{HF_REPO}")

## 8. Verify Upload

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(HF_REPO, repo_type="model")
print(f"Files in {HF_REPO}:")
for f in files:
    print(f"  - {f}")

print("\nYour backend can now download these automatically on startup.")
print(f"Set HF_REPO={HF_REPO} as an environment variable on your hosting platform.")